# 🔍 KOSTRA Datakontroller og Regnskapsanalyse

Denne Jupyter Notebook-en brukes i **VS Code** for å kjøre **automatiserte kvalitets- og verifikasjonskontroller** på KOSTRA-regnskapsdataene i `kostra.duckdb` og CSV-kildene.

### Kontrollpunkter:
1. **Integritetssjekk:** Sjekker at alle kontokoder og funksjoner finnes i dimensjonstabellene.
2. **Balanse- og Avvikskontroll:** Identifiserer vesentlige merforbruk og uvanlige posteringer.
3. **Velocity- / Endringssjekk:** Belyser kontoer med brå månedlige økninger i vikar- og overtidskostnader.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import os

# Koble til den lokale DuckDB-databasen
db_candidates = ['../data/database/kostra.duckdb', 'data/database/kostra.duckdb']
db_path = next((p for p in db_candidates if os.path.exists(p)), ':memory:')
con = duckdb.connect(db_path)
print(f'✅ Koblet til DuckDB ({db_path})')

## 1. Datakvalitet og Relasjonskontroll (Data Integrity)

In [ ]:
# Sjekk 1: Manglende kobling mot Kontoplan
query_unmapped_konto = '''
SELECT DISTINCT f.Kontokode_6Siffer
FROM fact_data f
LEFT JOIN dim_kostra_kontoplan k 
  ON CAST(f.Kontokode_6Siffer AS VARCHAR) = CAST(k.Kontokode_6Siffer AS VARCHAR)
WHERE k.Kontokode_6Siffer IS NULL;
'''
unmapped = con.query(query_unmapped_konto).df()
if len(unmapped) == 0:
    print('🟢 KONTROLL BESTÅTT: Alle regnskapskontoer finnes i dim_kostra_kontoplan!')
else:
    print(f'🔴 ADVARSEL: Funnet {len(unmapped)} ukjente kontokoder!')
    display(unmapped)

## 2. Finansiell Avviks- og Terskelkontroll

In [ ]:
# Hent topp 10 største merforbruk fra analytisk view
query_avvik = '''
SELECT 
    Registrert_Dato, Sektor, Funksjon_Kode, Funksjon_Navn,
    Konto_Kode, Kontonavn, Ansvar_Navn,
    Sum_Budsjett, Sum_Regnskap, Avvik_Kr, Forbruk_Pct, Status_Indikator
FROM vw_avvik_summary
WHERE Avvik_Kr < 0
ORDER BY Avvik_Kr ASC
LIMIT 10;
'''
df_avvik = con.query(query_avvik).df()
display(df_avvik)

## 3. Ekstremverdier og Månedlig Akselerasjonskontroll (Velocity Check)

In [ ]:
# Sjekk månedlig økning på vikarer og overtid
query_velocity = '''
SELECT 
    Dato,
    Kontokode_6Siffer,
    Regnskap,
    LAG(Regnskap, 1) OVER (PARTITION BY Kontokode_6Siffer ORDER BY Dato) AS Forrige_Mnd,
    Regnskap - LAG(Regnskap, 1) OVER (PARTITION BY Kontokode_6Siffer ORDER BY Dato) AS Mnd_Endring
FROM fact_data
WHERE Kontokode_6Siffer IN ('102000', '102013', '102031', '104000')
ORDER BY Mnd_Endring DESC;
'''
df_vel = con.query(query_velocity).df()
display(df_vel.head(10))